In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import time

print("IP 2 started: computing the matching score (M)...")

# -------------------------------------------------------------------------
# STEP 1: LOAD THE DATA
# -------------------------------------------------------------------------
try:
    # Load the candidate's vectors (numpy array)
    candidate_vectors = np.load("IP1_output_candidate_vectors.npy")
    print(f"-> Candidate data loaded. Number of skills: {candidate_vectors.shape[0]}")

    # Load the project data (pandas dataframe)
    df_projects = pd.read_pickle("IP1_output_listings_vectorized.pkl")
    print(f"-> {len(df_projects)} projects loaded.")
    
except FileNotFoundError:
    print("ERROR: IP 1 output files not found. Please complete IP 1 first.")
    raise

# -------------------------------------------------------------------------
# STEP 2: SCORING FUNCTION (COSINE SIMILARITY)
# -------------------------------------------------------------------------
def calculate_matching_score(project_vectors, candidate_vectors):
    """
    Computes how well the candidate's skills cover the project's requirements.
    Method: for each project requirement, find the candidate's best skill and take the mean.
    """
    # Return 0 if the project's skill vector is empty or invalid
    if project_vectors is None or len(project_vectors) == 0:
        return 0.0
    
    # Shape check (make it 2D if a 1D array is given)
    if len(project_vectors.shape) == 1:
        project_vectors = project_vectors.reshape(1, -1)
        
    # 1. Compute the similarity matrix: (number of project skills x number of candidate skills)
    # E.g. if the project has 5 skills and the candidate 13 -> a (5x13) matrix
    similarity_matrix = cosine_similarity(project_vectors, candidate_vectors)
    
    # 2. For each project requirement, find the candidate skill that covers it BEST (max)
    # axis=1: take the maximum of each row (project skill)
    best_matches = similarity_matrix.max(axis=1)
    
    # 3. Take the mean (how much of the project is 'covered')
    # Convert the 0-1 value to a 0-100 score
    score = best_matches.mean() * 100
    
    return score

# -------------------------------------------------------------------------
# STEP 3: SCORE ALL PROJECTS
# -------------------------------------------------------------------------
print("\nScoring projects (this may take a few seconds)...")
start_time = time.time()

# Compute the score for each row (project)
df_projects["M_Score"] = df_projects["skill_vectors"].apply(
    lambda p_vec: calculate_matching_score(p_vec, candidate_vectors)
)

end_time = time.time()
print(f"-> Scoring completed ({end_time - start_time:.2f} seconds).")

# -------------------------------------------------------------------------
# STEP 4: SORT AND SHOW THE RESULTS
# -------------------------------------------------------------------------
# Sort by M_Score in descending order
df_result = df_projects.sort_values(by="M_Score", ascending=False)

print("\n🏆 TOP 10 PROJECTS FOR THE CANDIDATE (by M-Score) 🏆")
print("-" * 80)

# Show the results (we use the link instead of the title)
for i, row in df_result.head(10).iterrows():
    # Try to guess the project name from the link (last part of the URL)
    project_estimated_name = row['link'].split('/')[-1].replace('_en', '').replace('-', ' ').upper()
    
    print(f"{i+1}. [Score: {row['M_Score']:.2f}] {project_estimated_name}")
    print(f"   Link: {row['link']}")
    print(f"   Matched skills (sample): {row['extracted_skills'][:5]}...") 
    print("-" * 80)

# Save the results (to be used in IP 5)
df_result.to_pickle("IP2_output_listings_M_scored.pkl")
print("\nFile saved: 'IP2_output_listings_M_scored.pkl'")

IP 2 started: computing the matching score (M)...
-> Candidate data loaded. Number of skills: 13
-> 952 projects loaded.

Scoring projects (this may take a few seconds)...
-> Scoring completed (0.23 seconds).

🏆 TOP 10 PROJECTS FOR THE CANDIDATE (by M-Score) 🏆
--------------------------------------------------------------------------------
149. [Score: 99.87] 48828
   Link: https://youth.europa.eu/solidarity/opportunity/48828_en
   Matched skills (sample): ['social medium', 'lo', 'graphic design', 'e', 'persona']...
--------------------------------------------------------------------------------
348. [Score: 99.86] 47877
   Link: https://youth.europa.eu/solidarity/opportunity/47877_en
   Matched skills (sample): ['english', 'social medium', 'com', 'usp', 'food']...
--------------------------------------------------------------------------------
204. [Score: 99.86] 48664
   Link: https://youth.europa.eu/solidarity/opportunity/48664_en
   Matched skills (sample): ['communication', 'sans'